# Phase 1: Training Pipeline — EfficientNet-B3 Fine-tuning

**Task**: Conifer Family Classification (Task A)

This notebook trains an EfficientNet-B3 model using fine-tuning with differential learning rates on 4 conifer families.

**Strategy** (per WoodID_plan.md):
- Backbone LR = 1e-5 (1/10 of head LR) — preserves pretrained ImageNet features
- Head LR = 1e-4
- 50 epochs, cosine annealing, label smoothing 0.1

---

## Contents

1. [Setup & Root Detection](#1-setup)  
2. [Install Dependencies](#2-install-dependencies)  
3. [Download Dataset from HuggingFace](#3-download-dataset)  
4. [Verify Dataset](#4-verify-dataset)  
5. [Create Dataloaders](#5-create-dataloaders)  
6. [Create & Test Model](#6-create-test-model)  
7. [Fine-tune EfficientNet-B3](#7-fine-tune)  
8. [Evaluate on Test Set](#8-evaluate)

## 1. Setup & Root Detection <a id='1-setup'></a>

Detects whether running on Colab or local machine, and sets `ROOT` accordingly.

- **Colab**: `ROOT = /content/woodbench`
- **Local**: `ROOT = <repo-root>` (auto-detected from notebook location)

In [ ]:
!pip install huggingface_hub
from google.colab import userdata
import os
from huggingface_hub import login
from huggingface_hub import snapshot_download

hftoken = userdata.get('HF_key')
login(token= hftoken)

os.environ["GH_TOKEN"] = userdata.get('GH_TOKEN')

!git clone https://$GH_TOKEN@github.com/ManifoldsTeam/woodbench.git

In [ ]:
import sys
from pathlib import Path

# Auto-detect ROOT: Colab uses /content/, local uses repo root
if Path('/content').exists():
    # Running on Colab
    ROOT = Path('/content/woodbench')
    print(f"[Colab] ROOT = {ROOT}")
else:
    # Running locally — infer repo root from notebook location
    # notebooks/phase 1/...ipynb → repo root
    ROOT = Path(__file__).parent.parent if '__file__' in dir() else Path.cwd()
    print(f"[Local] ROOT = {ROOT}")

# Add repo root to sys.path so `from src import ...` works
sys.path.insert(0, str(ROOT))

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Install Dependencies <a id='2-install-dependencies'></a>

Only run this cell on Colab. Skip if already installed locally.

In [ ]:
# Only run on Colab
# if Path('/content').exists():
#     !pip install -q -r {ROOT}/requirements.txt

## Mount Drive:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Download Dataset from HuggingFace <a id='3-download-dataset'></a>

Downloads `IC4SD-Wood-Conifer20v1` from HuggingFace Hub to `/data/split/`.
Only downloads if data is not already present.

In [ ]:
from pathlib import Path

DATA_DIR = ROOT / "data" / "split"

if not DATA_DIR.exists():
    from huggingface_hub import snapshot_download
    print("Downloading dataset from HuggingFace...")
    snapshot_download(
        repo_id="ny0g507h3p/woodbench_A",
        repo_type="dataset",
        local_dir=str(DATA_DIR),
    )
else:
    print(f"Dataset already exists at {DATA_DIR}")

print(f"DATA_DIR = {DATA_DIR}")

In [ ]:
%cd woodbench

In [ ]:
!git checkout feat/task-A

In [ ]:
!ls -a data/split

## 4. Verify Dataset <a id='4-verify-dataset'></a>

In [ ]:
from src.data import ConiferDataset

# Check structure
print("Dataset structure:")
for split in ['train', 'val', 'test']:
    split_path = DATA_DIR / split
    if split_path.exists():
        families = [d.name for d in split_path.iterdir() if d.is_dir()]
        print(f"  {split}/: {len(families)} families")

# Load train set and show distribution
train_ds = ConiferDataset(DATA_DIR, split='train', task='taskA')
print(f"\nTotal training samples: {len(train_ds)}")
print(f"Classes: {train_ds.classes}")
print(f"\nClass distribution:")
for family, count in train_ds.get_class_distribution().items():
    print(f"  {family}: {count} ({count/len(train_ds)*100:.1f}%)")

## 5. Create Dataloaders <a id='5-create-dataloaders'></a>

In [ ]:
from src.data import get_dataloaders

BATCH_SIZE = 32  # Lower batch size for EfficientNet-B3 (larger model)
IMG_SIZE = 300   # EfficientNet-B3 native resolution
NUM_WORKERS = 4

train_loader, val_loader, test_loader = get_dataloaders(
    data_root=DATA_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    img_size=IMG_SIZE,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Verify a batch
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

## 6. Create & Test Model <a id='6-create-test-model'></a>

In [ ]:
from src.models import create_model, BACKBONE_NAMES

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Check available backbones
print("\nAvailable backbones:")
for name, timm_name in BACKBONE_NAMES.items():
    print(f"  {name}: {timm_name}")

# Create model (unfrozen for fine-tuning)
model = create_model('efficientnet_b3', num_classes=4, pretrained=True, freeze_backbone=False)
model = model.to(device)

# Test forward pass
outputs = model(images.to(device))
print(f"\nOutput shape: {outputs.shape}  (expected: ({images.shape[0]}, 4))")

# Verify trainable parameters
trainable = {n for n, p in model.named_parameters() if p.requires_grad}
frozen   = {n for n, p in model.named_parameters() if not p.requires_grad}
print(f"\nTrainable params: {len(trainable)}")
print(f"Frozen params:    {len(frozen)}")

## 7. Fine-tune EfficientNet-B3 <a id='7-fine-tune'></a>

Backbone LR = 1e-5, Head LR = 1e-4, 50 epochs, cosine annealing, AMP.

Only the best checkpoint is saved (`save_best_only=True`).

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive")
EXP_DIR = DRIVE_ROOT / "woodbench_experiments" / "taskA" / "efficientnet_b3"

EXP_DIR.mkdir(parents=True, exist_ok=True)

print(EXP_DIR)

In [ ]:
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function
from src.utils import set_seed

set_seed(42)

# Re-create model (fresh init to start from pretrained)
model = create_model('efficientnet_b3', num_classes=4, pretrained=True, freeze_backbone=False)
model = model.to(device)

# Differential learning rates
HEAD_LR     = 1e-4
BACKBONE_LR = 1e-5
optimizer = create_optimizer(
    model,
    lr=HEAD_LR,
    weight_decay=1e-4,
    backbone_lr=BACKBONE_LR,
)

# Verify param groups
for pg in optimizer.param_groups:
    print(f"  [{pg['name']}] lr={pg['lr']:.2e}, {len(pg['params'])} tensors")

scheduler = create_scheduler(optimizer, epochs=50, scheduler_type='cosine')
criterion = create_loss_function(loss_type='label_smoothing', label_smoothing=0.1)

engine = TrainingEngine(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=device,
    use_amp=True,
)

EXP_DIR = DRIVE_ROOT / "woodbench_experiments" / "taskA" / "efficientnet_b3"
EXP_DIR.mkdir(parents=True, exist_ok=True)
print(EXP_DIR)

print("\nStarting fine-tuning (50 epochs)...")
results = engine.fit(
    epochs=50,
    output_dir=EXP_DIR,
    name='efficientnet_b3_finetune',
    save_best_only=True,
    verbose=True,
)

print(f"\nBest val accuracy: {results['best_val_acc']:.2f}%")

In [ ]:
!ls -a /content/drive/MyDrive/woodbench_experiments/taskA/efficientnet_b3/

## 8. Evaluate on Test Set <a id='8-evaluate'></a>

Loads the best checkpoint and evaluates on the held-out test set.

In [ ]:
from src.eval import compute_metrics, format_metrics, save_confusion_matrix

CLASS_NAMES = ['Cupressaceae', 'Pinaceae', 'Podocarpaceae', 'Taxaceae']

# Load best checkpoint
checkpoint_path = EXP_DIR / "efficientnet_b3_finetune_best.pth"
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded checkpoint from epoch {checkpoint['epoch']+1}")

# Evaluate
metrics = compute_metrics(
    model=model,
    data_loader=test_loader,
    device=device,
    num_classes=4,
    class_names=CLASS_NAMES,
)

print(format_metrics(metrics, CLASS_NAMES))

# Save confusion matrix
save_confusion_matrix(
    cm=metrics['confusion_matrix'],
    class_names=CLASS_NAMES,
    output_path=EXP_DIR / "confusion_matrix.png",
    title="Confusion Matrix — EfficientNet-B3 Fine-tune (Test Set)",
)
print(f"Confusion matrix saved to {EXP_DIR / 'confusion_matrix.png'}")